In [22]:
from pathlib import Path
from PIL import Image
import os, json, shutil
import matplotlib.pyplot as plt
import numpy as np

from utils.preprocessing import find_stop_line, score_page
from utils.ocr_utils import extract_text_lines, convert_file, pdf2png, convert_from_label_studio, convert_to_label_studio
from utils.labeling import group_lines_by_row, label_row, flatten_labels

# Raw PDF
RAW_PDF = "data/raw_pdfs"
# Direct OCR Outputs
OCR_DIR = "data/ocr_json"
OCR_IMG = "data/ocr_img"
# PNG
PAGES_DIR = "data/bill_png"
# Training Data
TRAIN_IMG = "data/training_data/images"
TRAIN_ANN = "data/training_data/line_level_tokens"
TRAIN_WRD = "data/training_data/word_level_tokens"
# File Backups
LABEL_STD = "data/training_data/label_studio"

## Step 1: OCR ##
The first step is to download paddleOCR and test it on some of the dataset. PaddleOCR is ran with *PaddlePaddle* as their deep learning model so it's important to download both libraries together

Source: https://github.com/PaddlePaddle/PaddleOCR

For CPU-only PaddlePaddle:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

In [23]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,)

c:\Users\alexr\anaconda3\envs\ml_beam\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\alexr\anaconda3\envs\ml_beam\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_rec`.


### 1.1 Convert PDF to PNG

In [25]:
pdf_path = Path(RAW_PDF) / "electric bills-part-6.pdf"
pdf2png(pdf_path, PAGES_DIR, dpi=200);

Saved page image: data\bill_png\electric bills-part-6_0.png
Saved page image: data\bill_png\electric bills-part-6_1.png
Saved page image: data\bill_png\electric bills-part-6_2.png


### 1.2 Use OCR Prediction Model

In [26]:
for img_path in sorted(Path(PAGES_DIR).glob("electric bills-part-6*.png")):
    result = ocr.predict(input=str(img_path))
    for page in result:
        page.save_to_json(OCR_DIR)
        page.save_to_img(OCR_IMG)

## Step 2: Data Preprocessing ##
We've now saved our page into their folders pages and outputs, saving the images and json files respectively. <br><br>Utility companies send out bills with multiple pages of information. It would be a waste to train our model on all those pages where classification isn't needed. Similar to **Term Frequency-Inverse Document Frequency (TF-IDF)**, each page will be read to count through how many lines, words, and digits to determine which page is likely to have the itemized bill. We will move this file into its own folder to train and test on.

### 2.1 Page Classification
Determine confidence scores of each line per page, and word density per page

In [27]:
# Save score values per page
page_scores = []
confidence_scores = {}

# Loop through each page in the output
for fname in sorted(os.listdir(OCR_DIR)):
    path = Path(OCR_DIR) / fname
    # Open file to read
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f) 
    # Retrieve confidence scores of all 3 pages together
    scores = data.get("rec_scores", [])
    if scores:
        confidence_scores[fname] = scores
    lines = extract_text_lines(data)
    stop_idx = find_stop_line(lines)
    lines = lines[:stop_idx]
    keyword_density, numeric_density = score_page(lines)
    # Print page densities
    print(f"{fname}: lines={len(lines)}, keyword_density={keyword_density:.4f}, numeric_density={numeric_density:.4f}")
    page_scores.append({
        "fname": fname,
        "keyword_density": keyword_density,
        "numeric_density": numeric_density,
    })
    
# Select best page
best_page = max(page_scores, key=lambda x: x["keyword_density"])

electric bills-part-6_0_res.json: lines=116, keyword_density=0.0172, numeric_density=0.1690
electric bills-part-6_1_res.json: lines=87, keyword_density=0.0698, numeric_density=0.1977
electric bills-part-6_2_res.json: lines=9, keyword_density=0.0000, numeric_density=0.2000


### 2.2 Plotting Page Classification Metrics

Below are some plots to show our metrics and results from paddleOCR. Checking our confidence scores is cricual to determining what values to use and whittle out the lines incorrectly recongized, while comparing density scores allows us to determine how we picked our best page

In [ ]:
# Rule out specific lines with low confidence scores to avoid noisy details
plt.figure(figsize=(5, 4))
for fname, scores in confidence_scores.items():
  plt.hist(scores, bins=200, range=(0, 1), alpha = 0.5, edgecolor="black", label=fname, linewidth=0.5)
plt.title(f"OCR Confidence: All Pages")
plt.xlabel("Confidence Score")
plt.ylabel("# of Lines")
plt.axvline(x=0.925, color="red", linestyle="--", label="0.925 threshold")
plt.legend()
plt.show() 

# Retrieve metrics
page_names = [f"Page {i+1}" for i in range(len(page_scores))]
keyword_densities = [s["keyword_density"] for s in page_scores]
numeric_densities = [s["numeric_density"] for s in page_scores]
# Create x axis and bar width
x = np.arange(len(page_names))
width = 0.25
# Create bar plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(x - width/2, keyword_densities, width, label="Keyword Density", color="steelblue")
ax.bar(x + width/2, numeric_densities, width, label="Numeric Density", color="orange")
ax.set_ylabel("Density Score")
ax.set_xticks(x)
ax.set_xticklabels(page_names, rotation=15, ha="right")
ax.set_title("Page Selection Metrics")
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Page Selection
After determining which page is the one holding our bill. We will move the base png image to train our model on

In [28]:
# Maintain base form of file to move
base = best_page['fname'].replace("_res.json","")
ann_src = Path(OCR_DIR) / best_page['fname']

if best_page:
  # Move JSON to annotations
  ann_dst = Path(TRAIN_ANN) / best_page['fname']
  shutil.copy2(ann_src, ann_dst)
  # Move best page from PDF to image first
  img_src = Path(PAGES_DIR) / (base + ".png")
  img_dst = Path(TRAIN_IMG) / (base + ".png")
  shutil.copy2(img_src, img_dst)
  print(f"Copied image: {img_dst}")

Copied image: data\training_data\images\electric bills-part-6_1.png


### 2.4 Prep JSON File for Labeling
Now it's crucial we build our labels for LayoutLM accordingly

In [29]:
img_path = Path(TRAIN_IMG) / (base + ".png")
# Determine size
with Image.open(img_path) as img:
    page_width, page_height = img.size
input_file = Path(TRAIN_ANN) / best_page['fname']

# Convert to LayoutLM ready format
entries = convert_file(input_file)
rows = group_lines_by_row(entries)
labeled_rows = [label_row(row) for row in rows]
page_entry = flatten_labels(
    page_id = base,
    image_path = (Path(TRAIN_IMG) / (base + ".png")).as_posix(),
    labeled_rows= labeled_rows,
    page_width= page_width,
    page_height= page_height
)

out_path = Path(TRAIN_WRD) / (base + ".json" )
with out_path.open("w", encoding="utf-8") as fh:
    fh.write(json.dumps(page_entry, ensure_ascii=False, indent=2) + "\n")

# Print token/label pairs for inspection
print(f"\n{'TOKEN':<20} {'LABEL'}")
print("-" * 35)
for token, label in zip(page_entry["tokens"], page_entry["labels"]):
    # only print non-O labels to reduce noise
    if label != "O":  
        print(f"{token:<20} {label}")


TOKEN                LABEL
-----------------------------------
31                   B-KWH_USAGE
$4,495.10            B-KWH_COST
$5,260.98            B-TOTAL_COST
45600                B-KWH_USAGE
4,240.66             B-KWH_COST
208.0                B-KW_USAGE
529.78               B-KW_COST


### 2.5 Intermediate Step: Manual Label Studio Corrections ###

**This is not necessary to run. Not meant to be run for CSE 498 Final Project Submission** Currently my labeling scheme can only get me so far. To ensure correct training is done, we can use label-studio to finish labeling our pipeline

In [ ]:
convert_to_label_studio(jsonl_path = Path(TRAIN_WRD) / (base + ".json"),
                        output_path = Path(LABEL_STD) / (base + "label_studio_import.json"),
                        image_root = TRAIN_IMG
)

In [ ]:
convert_from_label_studio(export_path  = Path(LABEL_STD) / (base + "label_studio_import.json"),
                          original_path = Path(TRAIN_WRD) / (base + ".json"),
                          output_path  = Path(TRAIN_WRD) / (base + ".json")
)

In [ ]:
# Copy folders from data to log folder ## CREATED BY CHATGPT ##
def archive_to_logs(logs_dir="logs"):
    folders = [
        ("data/pages",                                 "logs/pages"),
        ("data/ocr_pages",                             "logs/ocr_pages"),
        ("data/ocr_output",                            "logs/ocr_output"),
        ("data/training_data/annotation_raw",         "logs/training_data/annotation_raw"),
        ("data/training_data/annotation_words",       "logs/training_data/annotation_words"),
        ("data/training_data/images",                  "logs/training_data/images"),
        ("data/training_data/label_studio",            "logs/training_data/label_studio"),
    ]

    for src, dst in folders:
        Path(dst).mkdir(parents=True, exist_ok=True)
        for file in Path(src).iterdir():
            if file.is_file():
                shutil.move(file, Path(dst) / file.name)
                print(f"Copied: {file.name} → {dst}")

archive_to_logs()

## Step 3: LayoutLMv3 ##
Now comes our model we are going to use for transfer learning. LayoutLM extends the traditional trasnformer model by integrating layout information into the input embeddings designed for document AI tasks that can model texts, images and layout together. It will be the backbone of our project. Now that we have filtered our data into each respective folder, let's pass it into our model to fine-tune it.
### 3.1 Import Statements

In [ ]:
from transformers import AutoProcessor, AutoModelForTokenClassification
import torch, torchvision
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import classification_report as sk_classification_report

### 3.2 Load Model and Test CPU/GPU

In [ ]:
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

print(f"Torch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

### 3.3 Initialize Classification Model
We want to create our labels and transform them into integers to match standard HuggingFace datasets

In [ ]:
# Create labels list
LABELS = ["O", "B-KWH_USAGE", "B-KWH_COST", "B-KW_USAGE", "B-KW_COST", "B-TOTAL_COST"]
# Match HuggingFace Dataset
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}
# Initialize model with correct number of labels
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels  = len(LABELS),
    label2id    = label2id,
    id2label    = id2label,
    ignore_mismatched_sizes = True
)
print(f"Model initialized with {len(LABELS)} labels")

### 3.4 Load Training and Testing Data
Used an 80/20 split to train data

In [ ]:
# Store corrected training samples
training_samples = []

for path in sorted(Path(TRAIN_WRD).glob("*.json")):
  # Load corrected annotations
  with open(path, "r") as f:
      training_samples.append(json.load(f))
      
print(f"Total training samples: {len(training_samples)}")
# Determine split
train_samples, val_samples = train_test_split(
  training_samples,
  test_size=0.2,
  random_state=42
)
print(f"Train: {len(train_samples)}")
print(f"Val:   {len(val_samples)}")

### 3.5 Encoding Function

In [ ]:
def process_sample(sample, processor, label2id):
    # Load image sample to store
    image = Image.open(sample["image"]).convert("RGB")
    # Load ids
    label_ids = [label2id[label] for label in sample["labels"]]
    # Encode samples
    encoding = processor(
        images = image,
        text = sample["tokens"],
        boxes = sample["bboxes"],
        word_labels = label_ids,
        truncation = True,
        max_length = 512,
        padding = "max_length",
        return_tensors = "pt"
    )
    return encoding
# Store encodings for each part
train_encodings = [process_sample(s, processor, label2id) for s in train_samples]
val_encodings = [process_sample(s, processor, label2id) for s in val_samples]
print(f"Processed {len(train_encodings)} train, {len(val_encodings)} val")

### 3.6 Dataset Class
To encorrectly encode our datasets we want to create a standard dataset class. Stores sample ready for training or evaluation

In [ ]:
# Copied datacamp structure
class BillDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings[idx]["input_ids"].squeeze(),
            "attention_mask": self.encodings[idx]["attention_mask"].squeeze(),
            "bbox": self.encodings[idx]["bbox"].squeeze(),
            "pixel_values": self.encodings[idx]["pixel_values"].squeeze(),
            "labels": self.encodings[idx]["labels"].squeeze(),
        }
# Store dataset
train_dataset = BillDataset(train_encodings)
val_dataset = BillDataset(val_encodings)
# Store Dataloader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=2)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

### 3.7 Training & Validation

In [ ]:
# Handle GPU runs (later when I move to my other pc)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create model, optimizer and # of epochs
model = model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
epochs = 10
# Total Losses
train_losses = []
val_losses = []

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids = batch["input_ids"].to(device),
            attention_mask = batch["attention_mask"].to(device),
            bbox = batch["bbox"].to(device),
            pixel_values = batch["pixel_values"].to(device),
            labels = batch["labels"].to(device)
        )
        # Loss is automatically created from LayoutLM using CrossEntropy loss
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    # Validation
    all_true_labels = []
    all_pred_labels = []
    # Eval Mode
    model.eval()
    val_loss = 0
    # Metrics to save true and predicted labels
    flat_true = []
    flat_pred = []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                bbox = batch["bbox"].to(device),
                pixel_values = batch["pixel_values"].to(device),
                labels = batch["labels"].to(device)
            )
            val_loss += outputs.loss.item()
            # Get Predictions
            predictions = outputs.logits.argmax(-1)
            labels = batch["labels"]
            for pred, true, mask in zip(predictions, labels, batch["attention_mask"]):
                for p, t, m in zip(pred, true, mask):
                    if m.item() == 0:
                        continue
                    if t.item() == -100:
                        continue
                    flat_pred.append(id2label[p.item()])
                    flat_true.append(id2label[t.item()])
    # Find average loss
    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    # Print Loss
    print(f"Epoch {epoch+1}/{epochs} — Train Loss: {avg_train:.4f} Val Loss: {avg_val:.4f}")

# After training loop - get metrics ## COPIED FROM CHATGPT ##
report = sk_classification_report(flat_true, flat_pred, output_dict=True)
labels = [l for l in report.keys() if "avg" not in l and l != "accuracy"]
precision = [report[l]["precision"] for l in labels]
recall = [report[l]["recall"]    for l in labels]
f1 = [report[l]["f1-score"]  for l in labels]

print(sk_classification_report(flat_true, flat_pred))

### 3.7.1 Plotting Training/Validation Loss and Classification Metrics

In [ ]:
# Plot losses
plt.figure(figsize=(5, 4))
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", color="steelblue")
plt.plot(range(1, epochs+1), val_losses,   label="Val Loss",   color="orange")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Plot Classification Metrics
x = np.arange(len(labels))
width = 0.20
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(x - width, precision, width, label="Precision", color="steelblue")
ax.bar(x, recall, width, label="Recall", color="orange")
ax.bar(x + width, f1, width, label="F1", color="green")
ax.set_ylabel("Score")
ax.set_title("Per Label Evaluation Metrics")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylim(0, 1.15)
ax.legend()
plt.tight_layout()
plt.show()

### 3.7.2 Save Model on local machine

In [ ]:
# Save model and processor
MODEL_DIR = "models/layoutlmv3-bills"
os.makedirs(MODEL_DIR, exist_ok=True)
model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
print(f"Model saved to {MODEL_DIR}")